In [ ]:
# ==============================================================================
# ETL PIPELINE: BRONZE TO SILVER LAYER
# Data Cleaning & Transformation for Analytics Readiness
# ==============================================================================

import os
import sys
import logging
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, year, month, quarter, dayofmonth,
    round, when, lit, current_timestamp, count, concat_ws
)
from delta.tables import DeltaTable

# ── Initialize Spark Session ────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("EcommercePipeline-BronzeToSilver") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# ── Setup Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
logger.info("="*80)
logger.info("STARTING: Bronze to Silver ETL Pipeline")
logger.info("="*80)

# ── Define Local Paths (Relative to project root) ─────────────────────────
# In production, these would point to cloud storage (Azure Data Lake, S3, etc.)
BASE_PATH = Path("./data")
BRONZE_PATH = str(BASE_PATH / "bronze")
SILVER_PATH = str(BASE_PATH / "silver")

# Create directories if they don't exist
BASE_PATH.mkdir(exist_ok=True)
Path(BRONZE_PATH).mkdir(exist_ok=True)
Path(SILVER_PATH).mkdir(exist_ok=True)

logger.info(f"Bronze path: {BRONZE_PATH}")
logger.info(f"Silver path: {SILVER_PATH}")

In [ ]:
# ── Step 2: Data Quality Assessment ────────────────────────────────────────

logger.info("\n" + "="*80)
logger.info("DATA QUALITY ASSESSMENT")
logger.info("="*80)

from pyspark.sql.functions import count as spark_count, isnan, isnull

# Check for nulls in each column
null_counts = df_raw.select([
    spark_count(when(isnull(col(c)), 1)).alias(c) for c in df_raw.columns
]).collect()[0]

logger.info("Null value counts by column:")
for col_name, null_count in null_counts.asDict().items():
    logger.info(f"  {col_name}: {null_count}")

# Duplicate check
duplicate_count = df_raw.count() - df_raw.dropDuplicates().count()
logger.info(f"Duplicate rows (on all columns): {duplicate_count}")

# Duplicate check on order_id (should be unique)
order_id_duplicates = df_raw.count() - df_raw.dropDuplicates(["order_id"]).count()
logger.info(f"Duplicate order_ids: {order_id_duplicates}")

In [ ]:
# ── Step 3: Data Cleaning & Transformation ────────────────────────────────

logger.info("\n" + "="*80)
logger.info("APPLYING DATA TRANSFORMATIONS")
logger.info("="*80)

df = df_raw.copy()

# 3.1 Remove exact duplicates (based on order_id to ensure uniqueness)
logger.info("Removing duplicate orders...")
df = df.dropDuplicates(["order_id"])
logger.info(f"Rows after duplicate removal: {df.count()}")

# 3.2 Fill null values with meaningful defaults
logger.info("Handling missing values...")
df = df.fillna({
    "city": "Unknown",
    "sub_category": "Unknown",
    "product_name": "Unknown",
    "payment_mode": "Unknown",
    "payment_status": "Unknown",
    "region": "Unknown",
    "quantity": 0,
    "unit_price": 0.0,
    "discount": 0.0,
    "sales": 0.0,
    "profit": 0.0
})

# 3.3 Parse date string from DD-MM-YYYY to proper date type
logger.info("Parsing and validating dates...")
df = df.withColumn(
    "order_date_parsed",
    to_date(col("order_date"), "dd-MM-yyyy")
)

# Drop rows with invalid dates
df = df.filter(col("order_date_parsed").isNotNull())
logger.info(f"Rows after date validation: {df.count()}")

# 3.4 Remove business logic violations
logger.info("Validating business logic...")
df = df.filter(
    (col("quantity") >= 0) &
    (col("unit_price") >= 0) &
    (col("sales") >= 0) &
    (col("profit") >= col("sales") * -0.5)  # Profit margin shouldn't exceed 50% loss
)
logger.info(f"Rows after business validation: {df.count()}")

# 3.5 Feature engineering for temporal analysis
logger.info("Creating temporal features...")
df = df \
    .withColumn("order_year", year(col("order_date_parsed"))) \
    .withColumn("order_month", month(col("order_date_parsed"))) \
    .withColumn("order_quarter", quarter(col("order_date_parsed"))) \
    .withColumn("order_day", dayofmonth(col("order_date_parsed")))

# 3.6 Create calculated columns for analytics
logger.info("Creating analytical features...")
df = df \
    .withColumn("net_sales",
        round(col("sales") * (1 - col("discount") / 100), 2)) \
    .withColumn("profit_margin_pct",
        when(col("sales") > 0,
             round(col("profit") / col("sales") * 100, 2)
        ).otherwise(lit(0.0))) \
    .withColumn("revenue_bucket",
        when(col("sales") >= 100000, lit("High"))
        .when(col("sales") >= 30000, lit("Medium"))
        .otherwise(lit("Low"))) \
    .withColumn("_silver_load_timestamp", current_timestamp()) \
    .withColumn("_silver_load_date", col("order_date_parsed"))

logger.info("Data transformation complete!")
logger.info(f"Final row count: {df.count()}")
logger.info("\nTransformed data schema:")
df.printSchema()
logger.info("\nSample transformed records:")
df.select("order_id", "order_date_parsed", "order_year", "net_sales", "profit_margin_pct", "revenue_bucket").show(5)

In [ ]:
# ── Step 5: ETL Pipeline Summary ───────────────────────────────────────────

logger.info("\n" + "="*80)
logger.info("ETL PIPELINE EXECUTION SUMMARY")
logger.info("="*80)

summary_stats = {
    "Bronze Input Rows": df_raw.count(),
    "Silver Output Rows": df_silver.count(),
    "Rows Removed (Quality Issues)": df_raw.count() - df_silver.count(),
    "Total Revenue": float(df_silver.agg({"sales": "sum"}).collect()[0][0]),
    "Total Profit": float(df_silver.agg({"profit": "sum"}).collect()[0][0]),
    "Average Order Value": float(df_silver.agg({"sales": "avg"}).collect()[0][0]),
    "Total Orders": df_silver.count()
}

logger.info("\nKey Metrics:")
for metric, value in summary_stats.items():
    if "Rows" in metric or "Orders" in metric:
        logger.info(f"  {metric}: {int(value):,}")
    elif "Revenue" in metric or "Profit" in metric or "Value" in metric:
        logger.info(f"  {metric}: ₹{value:,.2f}")
    else:
        logger.info(f"  {metric}: {value}")

logger.info("\nData Distribution:")
df_silver.groupBy("revenue_bucket").count().show()
df_silver.groupBy("region").agg({"sales": "sum"}).show()

logger.info("\n" + "="*80)
logger.info("✓ BRONZE TO SILVER ETL PIPELINE COMPLETED SUCCESSFULLY")
logger.info("="*80)

## Step 5: Pipeline Execution Summary

Generate summary metrics and quality report for the ETL pipeline.

In [ ]:
# ── Step 4: Write to Silver Layer (Delta Format) ───────────────────────────

logger.info("\n" + "="*80)
logger.info("WRITING TO SILVER LAYER")
logger.info("="*80)

silver_table_path = str(Path(SILVER_PATH) / "ecommerce_sales")

try:
    # Select only relevant columns for Silver layer
    columns_for_silver = [
        "order_id", "order_date_parsed", "customer_name", "region", "city",
        "category", "sub_category", "product_name", "quantity", "unit_price",
        "discount", "sales", "net_sales", "profit", "profit_margin_pct",
        "revenue_bucket", "payment_mode", "payment_status",
        "order_year", "order_month", "order_quarter", "order_day",
        "_silver_load_timestamp", "_silver_load_date"
    ]
    
    df_silver = df.select(columns_for_silver)
    
    # Write with Delta format and partitioning
    # Partitioning by year/month improves query performance for time-based analytics
    df_silver.write \
        .format("delta") \
        .partitionBy("order_year", "order_month") \
        .mode("overwrite") \
        .save(silver_table_path)
    
    logger.info(f"✓ Successfully written {df_silver.count()} rows to Silver layer")
    logger.info(f"  Location: {silver_table_path}")
    logger.info(f"  Format: Delta (Parquet)")
    logger.info(f"  Partitions: order_year, order_month")
    
except Exception as e:
    logger.error(f"✗ Error writing to Silver layer: {str(e)}")
    raise

## Step 4: Write Cleaned Data to Silver Layer

Save the transformed data to the Silver layer using Delta format with partitioning for optimized queries.

## Step 3: Data Cleaning & Transformation

Apply transformation logic to clean and standardize data for the Silver layer.

## Step 2: Data Quality Assessment

Check for data quality issues before transformation.

In [ ]:
# ── Step 1: Load Data from Bronze Layer ────────────────────────────────────

# Check if CSV exists in bronze folder
bronze_file = Path(BRONZE_PATH) / "ecommerce_sales_raw.csv"

if not bronze_file.exists():
    logger.warning(f"Bronze file not found at {bronze_file}")
    logger.info("Creating sample data for demonstration...")
    
    # Create sample data for testing
    sample_data = [
        ("ORD001", "15-01-2024", "Customer A", "North", "Delhi", "Electronics", "Phones", "iPhone 13", 1, 50000.0, 10.0, 45000.0, 5000.0, "Credit Card", "Completed"),
        ("ORD002", "16-01-2024", "Customer B", "South", "Bangalore", "Clothing", "Shirts", "Premium Shirt", 2, 2500.0, 0.0, 5000.0, 1000.0, "Debit Card", "Completed"),
        ("ORD003", "16-01-2024", "Customer C", "West", "Mumbai", "Electronics", "Laptops", "Dell Laptop", 1, 70000.0, 5.0, 66500.0, 8000.0, "Wallet", "Completed"),
        ("ORD004", "17-01-2024", "Customer D", "East", "Kolkata", "Books", "Fiction", "Novel", 3, 500.0, 0.0, 1500.0, 200.0, "UPI", "Completed"),
        ("ORD005", "17-01-2024", "Customer E", "North", "Chandigarh", "Electronics", "Headphones", "Sony WH1000", 1, 15000.0, 20.0, 12000.0, 1500.0, "Credit Card", "Completed"),
    ]
    
    schema = "order_id STRING, order_date STRING, customer_name STRING, region STRING, city STRING, category STRING, sub_category STRING, product_name STRING, quantity INT, unit_price DOUBLE, discount DOUBLE, sales DOUBLE, profit DOUBLE, payment_mode STRING, payment_status STRING"
    
    df_raw = spark.createDataFrame(sample_data, schema=schema)
    
    # Save to bronze
    df_raw.coalesce(1).write.csv(str(BRONZE_PATH), header=True, mode="overwrite")
    logger.info(f"Sample data created with {df_raw.count()} rows")
else:
    # Load from CSV
    df_raw = spark.read.csv(
        str(bronze_file),
        header=True,
        inferSchema=True
    )
    logger.info(f"Loaded data from {bronze_file}")

# Data validation
row_count = df_raw.count()
logger.info(f"Total rows loaded: {row_count}")

# Show schema
logger.info("Schema:")
df_raw.printSchema()

# Display sample records
logger.info("Sample records:")
df_raw.show(5, truncate=False)

## Step 1: Data Ingestion from Bronze Layer

Load raw e-commerce transaction data from the Bronze layer. Perform initial data validation and log key metrics.

# Bronze → Silver Layer: Data Cleaning & Transformation

This notebook demonstrates the first stage of the ETL pipeline where raw data is cleaned, validated, and transformed for analytics readiness.

**Medallion Architecture:** Bronze (raw) → Silver (cleaned) → Gold (aggregated)

**Processing Steps:**
- Read raw e-commerce transaction data from Bronze layer
- Apply data quality checks and handle missing values
- Standardize data types and column names
- Remove duplicates and invalid records
- Perform feature engineering for temporal analysis
- Write cleaned data to Silver layer with partitioning

**Dataset:** Indian e-commerce sales (15,000 transactions)